In [2]:
# ============================================================
# FINAL MODELS — THRESHOLD SENSITIVITY >=1 / >=2 / >=3
# PCA TRAIN-ONLY FINAL VERSION
# Robust version against duplicated y_pred/y_true columns
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TARGET_CANDIDATES = [
    Path("./data/target_col.csv"),
    Path("../data/target_col.csv"),
    Path("../../data/target_col.csv"),
]

TARGET_PATH = None
for p in TARGET_CANDIDATES:
    if p.exists():
        TARGET_PATH = p
        break

if TARGET_PATH is None:
    raise FileNotFoundError("Could not find target_col.csv. Adjust TARGET_PATH manually.")

PREDICTION_FILES = {
    "victimization": Path("./final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv"),
    "perpetration": Path("./final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv"),
    "overlap": Path(
        "./final_overlap/overlap_final/"
        "overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42/"
        "outputs/predictions_with_probs.csv"
    ),
}

OUTPUT_DIR = Path("./final_threshold_sensitivity_PCA_trainonly")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TARGET PATH:", TARGET_PATH.resolve())
for outcome, path in PREDICTION_FILES.items():
    print(outcome, path.exists(), path)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def detect_col(df, candidates):
    cols_lower = {str(c).lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]

    for c in df.columns:
        cl = str(c).lower()
        if any(cand.lower() in cl for cand in candidates):
            return c

    return None


def get_first_col(df, colname):
    """
    If df[colname] returns a DataFrame because duplicated column names exist,
    keep only the first column.
    """
    x = df[colname]
    if isinstance(x, pd.DataFrame):
        print(f"WARNING: duplicated column name '{colname}'. Using first occurrence.")
        x = x.iloc[:, 0]
    return x


def to_1d_numeric(x, name):
    """
    Converts Series/DataFrame/array to clean 1D numeric Series.
    """
    if isinstance(x, pd.DataFrame):
        print(f"WARNING: {name} is DataFrame with shape {x.shape}. Using first column.")
        x = x.iloc[:, 0]

    arr = np.asarray(x)

    if arr.ndim > 1:
        print(f"WARNING: {name} has shape {arr.shape}. Using first column.")
        arr = arr[:, 0]

    s = pd.Series(arr)
    s = pd.to_numeric(s, errors="coerce")
    return s


def binary_metrics(y_true, y_pred):
    y_true = to_1d_numeric(y_true, "y_true")
    y_pred = to_1d_numeric(y_pred, "y_pred")

    valid = y_true.notna() & y_pred.notna()

    y_true = y_true.loc[valid].astype(int).to_numpy()
    y_pred = y_pred.loc[valid].astype(int).to_numpy()

    y_true = (y_true > 0).astype(int)
    y_pred = (y_pred > 0).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = 2 * ppv * recall / (ppv + recall) if (ppv + recall) > 0 else np.nan

    return {
        "n": int(len(y_true)),
        "positives": int(np.sum(y_true == 1)),
        "negatives": int(np.sum(y_true == 0)),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "recall_sensitivity": recall,
        "specificity": specificity,
        "precision_ppv": ppv,
        "npv": npv,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_positive": f1,
        "accuracy": accuracy_score(y_true, y_pred),
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        "fnr": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
    }


def load_predictions(path, outcome):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    raw = pd.read_csv(path)

    print("\n--------------------------------------")
    print("Loading predictions:", outcome)
    print("Path:", path)
    print("Columns:", list(raw.columns))

    # Remove duplicated column names before detection, keeping first occurrence
    if raw.columns.duplicated().any():
        print("WARNING duplicated columns in prediction file:")
        print(raw.columns[raw.columns.duplicated()].tolist())
        raw = raw.loc[:, ~raw.columns.duplicated()].copy()

    idx_col = detect_col(raw, [
        "idx_original",
        "original_idx",
        "idx",
        "index"
    ])

    y_pred_col = detect_col(raw, [
        f"y_pred_{outcome}",
        "y_pred_victim",
        "y_pred_perp",
        "y_pred_perpetration",
        "y_pred_overlap",
        "y_pred_intersect",
        "y_pred",
        "prediction"
    ])

    y_prob_col = detect_col(raw, [
        f"y_prob_{outcome}",
        "y_prob_victim",
        "y_prob_perp",
        "y_prob_perpetration",
        "y_prob_overlap",
        "y_prob_intersect",
        "y_prob",
        "probability",
        "score"
    ])

    if idx_col is None or y_pred_col is None:
        raise ValueError(
            f"Could not detect idx/y_pred columns for {outcome}. "
            f"Columns found: {list(raw.columns)}"
        )

    out = pd.DataFrame({
        "idx_original": pd.to_numeric(get_first_col(raw, idx_col), errors="coerce"),
        "y_pred": pd.to_numeric(get_first_col(raw, y_pred_col), errors="coerce"),
    })

    if y_prob_col is not None:
        out["y_prob"] = pd.to_numeric(get_first_col(raw, y_prob_col), errors="coerce")

    out = out.dropna(subset=["idx_original", "y_pred"]).copy()
    out["idx_original"] = out["idx_original"].astype(int)
    out["y_pred"] = (out["y_pred"] > 0).astype(int)

    print("Detected idx:", idx_col)
    print("Detected y_pred:", y_pred_col)
    print("Detected y_prob:", y_prob_col)
    print("Loaded n:", len(out))
    print("Predicted positives:", int(out["y_pred"].sum()))
    print("Predicted negatives:", int((out["y_pred"] == 0).sum()))

    return out


# ------------------------------------------------------------
# Load target file
# ------------------------------------------------------------

target_df = pd.read_csv(TARGET_PATH)
target_df = target_df.copy()

# idx_original in prediction files corresponds to dataframe index used in notebooks
target_df["idx_original"] = target_df.index.astype(int)

print("\n======================================")
print("Target shape:", target_df.shape)
print("\nTarget columns:")
for i, c in enumerate(target_df.columns):
    print(i, c)

# Detect count columns
V_COL = detect_col(target_df, [
    "V.SUM.TOTAL",
    "V_SUM_TOTAL",
    "victimization_sum",
    "victim_sum",
    "v.sum",
    "v_sum"
])

P_COL = detect_col(target_df, [
    "P.SUM.TOTAL",
    "P_SUM_TOTAL",
    "perpetration_sum",
    "perp_sum",
    "p.sum",
    "p_sum"
])

print("\nDetected V_COL:", V_COL)
print("Detected P_COL:", P_COL)

if V_COL is None or P_COL is None:
    raise ValueError(
        "Could not detect V.SUM.TOTAL and/or P.SUM.TOTAL. "
        "Check printed target columns and set V_COL/P_COL manually."
    )

target_df[V_COL] = pd.to_numeric(target_df[V_COL], errors="coerce")
target_df[P_COL] = pd.to_numeric(target_df[P_COL], errors="coerce")

print("\nV counts:")
print(target_df[V_COL].value_counts(dropna=False).sort_index())

print("\nP counts:")
print(target_df[P_COL].value_counts(dropna=False).sort_index())


# ------------------------------------------------------------
# Run threshold sensitivity
# ------------------------------------------------------------

thresholds = [1, 2, 3]
rows = []
rowlevel_outputs = {}

for outcome, pred_path in PREDICTION_FILES.items():
    pred = load_predictions(pred_path, outcome)

    merged = pred.merge(
        target_df[["idx_original", V_COL, P_COL]],
        on="idx_original",
        how="left"
    )

    # Remove duplicated columns after merge, if any
    if merged.columns.duplicated().any():
        print("WARNING duplicated columns after merge:")
        print(merged.columns[merged.columns.duplicated()].tolist())
        merged = merged.loc[:, ~merged.columns.duplicated()].copy()

    print("\n======================================")
    print("Outcome:", outcome)
    print("n:", len(merged))
    print("Missing V/P:", merged[[V_COL, P_COL]].isna().sum().to_dict())
    print("y_pred shape:", np.asarray(get_first_col(merged, "y_pred")).shape)
    print("y_pred counts:")
    print(get_first_col(merged, "y_pred").value_counts(dropna=False))

    rowlevel_outputs[outcome] = merged.copy()

    for t in thresholds:
        if outcome == "victimization":
            y_alt = (merged[V_COL] >= t).astype(int)
            alt_definition = f"{V_COL} >= {t}"

        elif outcome == "perpetration":
            y_alt = (merged[P_COL] >= t).astype(int)
            alt_definition = f"{P_COL} >= {t}"

        elif outcome == "overlap":
            y_alt = ((merged[V_COL] >= t) & (merged[P_COL] >= t)).astype(int)
            alt_definition = f"{V_COL} >= {t} AND {P_COL} >= {t}"

        else:
            continue

        metrics = binary_metrics(
            y_alt,
            get_first_col(merged, "y_pred")
        )

        rows.append({
            "outcome": outcome,
            "severity_threshold": f">={t}",
            "alternative_definition": alt_definition,
            **metrics
        })


sensitivity_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

main_csv = OUTPUT_DIR / "threshold_sensitivity_ge1_ge2_ge3_final_PCA_trainonly.csv"
pretty_csv = OUTPUT_DIR / "threshold_sensitivity_ge1_ge2_ge3_final_PCA_trainonly_pretty.csv"

sensitivity_df.to_csv(main_csv, index=False)

for outcome, merged in rowlevel_outputs.items():
    merged.to_csv(
        OUTPUT_DIR / f"{outcome}_threshold_sensitivity_rowlevel.csv",
        index=False
    )

# Pretty percent columns
pretty = sensitivity_df.copy()
metric_cols = [
    "recall_sensitivity",
    "specificity",
    "precision_ppv",
    "npv",
    "balanced_accuracy",
    "f1_positive",
    "accuracy",
    "fpr",
    "fnr"
]

for c in metric_cols:
    pretty[c] = pretty[c].apply(
        lambda x: f"{x*100:.1f}%" if pd.notna(x) else ""
    )

pretty.to_csv(pretty_csv, index=False)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 2400)

display_cols = [
    "outcome",
    "severity_threshold",
    "alternative_definition",
    "n",
    "positives",
    "negatives",
    "TP",
    "FP",
    "TN",
    "FN",
    "recall_sensitivity",
    "specificity",
    "precision_ppv",
    "npv",
    "balanced_accuracy",
    "f1_positive",
    "accuracy",
    "fpr",
    "fnr"
]

print("\n=== THRESHOLD SENSITIVITY >=1 / >=2 / >=3 ===")
print(
    sensitivity_df[display_cols].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print("\n=== PRETTY ===")
print(pretty[display_cols].to_string(index=False))

print("\nSaved main CSV to:")
print(main_csv.resolve())

print("\nSaved pretty CSV to:")
print(pretty_csv.resolve())

print("\nSaved row-level files to:")
print(OUTPUT_DIR.resolve())

TARGET PATH: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/data/target_col.csv
victimization True final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv
perpetration True final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv
overlap True final_overlap/overlap_final/overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42/outputs/predictions_with_probs.csv

Target shape: (4024, 12)

Target columns:
0 VÍCTIMA
1 PERPETRADOR
2 VICTIMA_PERPETRADOR
3 POLIVICTIMIZACION
4 POLIPERPETRACION
5 SOLO.VICTIMA
6 SOLO.PERPETRADOR
7 NO.VICT_NO.PERP
8 V.O
9 P.SUM.TOTAL
10 V.SUM.TOTAL
11 idx_original

Detected V_COL: V.SUM.TOTAL
Detected P_COL: P.SUM.TOTAL

V counts:
V.SUM.TOTAL
0.0     2011
1.0      723
2.0      378
3.0      278
4.0      176
5.0      111
6.0      102
7.0       62
8.0       52
9.0       37
10.0      24
11.0      18
12.0      15
13.0      12
14.0       5
15.0       7
16.0       2
17.0       3
18.0       2
19.0  

In [3]:
from pathlib import Path
import pandas as pd

path = Path("./final_threshold_sensitivity_PCA_trainonly/threshold_sensitivity_ge1_ge2_ge3_final_PCA_trainonly.csv")

print("Exists:", path.exists())
print("Path:", path.resolve())

df = pd.read_csv(path)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 2400)

print(df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

Exists: True
Path: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_threshold_sensitivity_PCA_trainonly/threshold_sensitivity_ge1_ge2_ge3_final_PCA_trainonly.csv
      outcome severity_threshold                alternative_definition   n  positives  negatives  TP  FP  TN  FN  recall_sensitivity  specificity  precision_ppv   npv  balanced_accuracy  f1_positive  accuracy   fpr   fnr
victimization                >=1                      V.SUM.TOTAL >= 1 942        458        484 355 379 105 103               0.775        0.217          0.484 0.505              0.496        0.596     0.488 0.783 0.225
victimization                >=2                      V.SUM.TOTAL >= 2 942        279        663 221 513 150  58               0.792        0.226          0.301 0.721              0.509        0.436     0.394 0.774 0.208
victimization                >=3                      V.SUM.TOTAL >= 3 942        204        738 164 570 168  40               0.80

In [4]:
# ============================================================
# DIAGNOSTIC — CHECK TARGET ALIGNMENT FOR THRESHOLD SENSITIVITY
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import confusion_matrix

TARGET_PATH = Path("./data/target_col.csv")

PREDICTION_FILES = {
    "victimization": Path("./final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv"),
    "perpetration": Path("./final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv"),
    "overlap": Path(
        "./final_overlap/overlap_final/"
        "overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42/"
        "outputs/predictions_with_probs.csv"
    ),
}

target_raw = pd.read_csv(TARGET_PATH).copy()
target_raw["raw_index"] = target_raw.index.astype(int)

V_COL = "V.SUM.TOTAL"
P_COL = "P.SUM.TOTAL"

target_raw[V_COL] = pd.to_numeric(target_raw[V_COL], errors="coerce")
target_raw[P_COL] = pd.to_numeric(target_raw[P_COL], errors="coerce")

def get_pred_cols(df):
    idx_col = [c for c in df.columns if c.lower() in ["idx_original", "idx", "index"]][0]
    y_true_col = [c for c in df.columns if c.startswith("y_true")][0]
    y_pred_col = [c for c in df.columns if c.startswith("y_pred")][0]
    return idx_col, y_true_col, y_pred_col

def cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"TP": tp, "FP": fp, "TN": tn, "FN": fn}

for outcome, path in PREDICTION_FILES.items():
    pred = pd.read_csv(path)
    idx_col, y_true_col, y_pred_col = get_pred_cols(pred)

    pred = pred[[idx_col, y_true_col, y_pred_col]].copy()
    pred = pred.rename(columns={
        idx_col: "idx_original",
        y_true_col: "y_true_official",
        y_pred_col: "y_pred"
    })

    pred["idx_original"] = pred["idx_original"].astype(int)
    pred["y_true_official"] = pred["y_true_official"].astype(int)
    pred["y_pred"] = pred["y_pred"].astype(int)

    # Candidate A: direct raw target row by idx_original
    direct = pred.merge(
        target_raw[["raw_index", V_COL, P_COL]],
        left_on="idx_original",
        right_on="raw_index",
        how="left"
    )

    if outcome == "victimization":
        y_direct = (direct[V_COL] >= 1).astype(int)
    elif outcome == "perpetration":
        y_direct = (direct[P_COL] >= 1).astype(int)
    else:
        y_direct = ((direct[V_COL] >= 1) & (direct[P_COL] >= 1)).astype(int)

    match_rate = (y_direct.to_numpy() == pred["y_true_official"].to_numpy()).mean()

    print("\n====================================")
    print("Outcome:", outcome)
    print("Prediction path:", path)
    print("n:", len(pred))
    print("Official y_true positives:", int(pred["y_true_official"].sum()))
    print("Direct target >=1 positives:", int(y_direct.sum()))
    print("Match official y_true vs direct target:", round(match_rate, 4))
    print("Official CM:", cm(pred["y_true_official"], pred["y_pred"]))
    print("Direct target CM:", cm(y_direct, pred["y_pred"]))

    mismatches = pred.loc[y_direct.to_numpy() != pred["y_true_official"].to_numpy()].copy()
    print("Mismatches:", len(mismatches))
    print(mismatches.head(10))


Outcome: victimization
Prediction path: final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv
n: 942
Official y_true positives: 465
Direct target >=1 positives: 458
Match official y_true vs direct target: 0.5234
Official CM: {'TP': np.int64(405), 'FP': np.int64(329), 'TN': np.int64(148), 'FN': np.int64(60)}
Direct target CM: {'TP': np.int64(355), 'FP': np.int64(379), 'TN': np.int64(105), 'FN': np.int64(103)}
Mismatches: 449
    idx_original  y_true_official  y_pred
1           1694                0       0
3            840                0       1
7            942                1       1
8           1579                1       1
9           1290                0       1
15          2055                1       1
17          3137                0       0
18          1904                0       1
21          1707                1       1
23          1670                1       0

Outcome: perpetration
Prediction path: final_perpetrator/content/perpetrator_v3/predictions_w

In [5]:
# ============================================================
# FIND CORRECT TARGET ALIGNMENT + THRESHOLD SENSITIVITY >=1/>=2/>=3
# PCA TRAIN-ONLY FINAL VERSION
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

FEATURES_PATH = Path("./data/lista_global_vars.csv")
TARGET_PATH = Path("./data/target_col.csv")

PREDICTION_FILES = {
    "victimization": Path("./final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv"),
    "perpetration": Path("./final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv"),
    "overlap": Path(
        "./final_overlap/overlap_final/"
        "overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42/"
        "outputs/predictions_with_probs.csv"
    ),
}

OUTPUT_DIR = Path("./final_threshold_sensitivity_PCA_trainonly_ALIGNED")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

V_COL = "V.SUM.TOTAL"
P_COL = "P.SUM.TOTAL"


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def detect_col(df, candidates):
    cols_lower = {str(c).lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]

    for c in df.columns:
        cl = str(c).lower()
        if any(cand.lower() in cl for cand in candidates):
            return c

    return None


def get_first_col(df, colname):
    x = df[colname]
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]
    return x


def load_predictions(path, outcome):
    raw = pd.read_csv(path)

    if raw.columns.duplicated().any():
        raw = raw.loc[:, ~raw.columns.duplicated()].copy()

    idx_col = detect_col(raw, ["idx_original", "original_idx", "idx", "index"])

    y_true_col = detect_col(raw, [
        f"y_true_{outcome}",
        "y_true_victim",
        "y_true_perp",
        "y_true_perpetration",
        "y_true_overlap",
        "y_true_intersect",
        "y_true",
        "target"
    ])

    y_pred_col = detect_col(raw, [
        f"y_pred_{outcome}",
        "y_pred_victim",
        "y_pred_perp",
        "y_pred_perpetration",
        "y_pred_overlap",
        "y_pred_intersect",
        "y_pred",
        "prediction"
    ])

    y_prob_col = detect_col(raw, [
        f"y_prob_{outcome}",
        "y_prob_victim",
        "y_prob_perp",
        "y_prob_perpetration",
        "y_prob_overlap",
        "y_prob_intersect",
        "y_prob",
        "probability",
        "score"
    ])

    if idx_col is None or y_true_col is None or y_pred_col is None:
        raise ValueError(f"Cannot detect columns for {outcome}: {list(raw.columns)}")

    out = pd.DataFrame({
        "idx_original": pd.to_numeric(get_first_col(raw, idx_col), errors="coerce"),
        "y_true_official": pd.to_numeric(get_first_col(raw, y_true_col), errors="coerce"),
        "y_pred": pd.to_numeric(get_first_col(raw, y_pred_col), errors="coerce"),
    })

    if y_prob_col is not None:
        out["y_prob"] = pd.to_numeric(get_first_col(raw, y_prob_col), errors="coerce")

    out = out.dropna(subset=["idx_original", "y_true_official", "y_pred"]).copy()
    out["idx_original"] = out["idx_original"].astype(int)
    out["y_true_official"] = (out["y_true_official"] > 0).astype(int)
    out["y_pred"] = (out["y_pred"] > 0).astype(int)

    return out


def make_y_from_counts(df, outcome, threshold=1):
    if outcome == "victimization":
        return (df[V_COL] >= threshold).astype(int)

    if outcome == "perpetration":
        return (df[P_COL] >= threshold).astype(int)

    if outcome == "overlap":
        return ((df[V_COL] >= threshold) & (df[P_COL] >= threshold)).astype(int)

    raise ValueError(outcome)


def cm_dict(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
    }


def binary_metrics(y_true, y_pred):
    y_true = pd.to_numeric(pd.Series(np.asarray(y_true).reshape(-1)), errors="coerce")
    y_pred = pd.to_numeric(pd.Series(np.asarray(y_pred).reshape(-1)), errors="coerce")

    valid = y_true.notna() & y_pred.notna()

    y_true = y_true.loc[valid].astype(int).to_numpy()
    y_pred = y_pred.loc[valid].astype(int).to_numpy()

    y_true = (y_true > 0).astype(int)
    y_pred = (y_pred > 0).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = 2 * ppv * recall / (ppv + recall) if (ppv + recall) > 0 else np.nan

    return {
        "n": int(len(y_true)),
        "positives": int(np.sum(y_true == 1)),
        "negatives": int(np.sum(y_true == 0)),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "recall_sensitivity": recall,
        "specificity": specificity,
        "precision_ppv": ppv,
        "npv": npv,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_positive": f1,
        "accuracy": accuracy_score(y_true, y_pred),
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        "fnr": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
    }


# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

features_raw = pd.read_csv(FEATURES_PATH).copy()
target_raw = pd.read_csv(TARGET_PATH).copy()

target_raw[V_COL] = pd.to_numeric(target_raw[V_COL], errors="coerce")
target_raw[P_COL] = pd.to_numeric(target_raw[P_COL], errors="coerce")

print("features_raw:", features_raw.shape)
print("target_raw:", target_raw.shape)

predictions = {
    outcome: load_predictions(path, outcome)
    for outcome, path in PREDICTION_FILES.items()
}

for outcome, pred in predictions.items():
    print("\n", outcome)
    print("n:", len(pred))
    print("idx min/max:", pred["idx_original"].min(), pred["idx_original"].max())
    print("official positives:", int(pred["y_true_official"].sum()))
    print("official CM:", cm_dict(pred["y_true_official"], pred["y_pred"]))


# ------------------------------------------------------------
# Build candidate alignments
# ------------------------------------------------------------

candidates = {}

# Candidate 1: raw target direct index
c1 = target_raw[[V_COL, P_COL]].copy()
c1["alignment_key"] = np.arange(len(c1))
candidates["raw_direct_index"] = c1

# Candidate 2: complete cases in features + target, original index
combo = pd.concat(
    [
        features_raw.reset_index(drop=True),
        target_raw[[V_COL, P_COL]].reset_index(drop=True)
    ],
    axis=1
)

complete_mask = combo.replace([np.inf, -np.inf], np.nan).notna().all(axis=1)

c2 = target_raw.loc[complete_mask, [V_COL, P_COL]].copy()
c2["alignment_key"] = c2.index.astype(int)
candidates["complete_cases_original_index"] = c2

# Candidate 3: complete cases in features + target, reset index
c3 = target_raw.loc[complete_mask, [V_COL, P_COL]].copy().reset_index(drop=True)
c3["alignment_key"] = np.arange(len(c3))
candidates["complete_cases_reset_index"] = c3

# Candidate 4: complete cases in features only, original index
features_complete_mask = features_raw.replace([np.inf, -np.inf], np.nan).notna().all(axis=1)

c4 = target_raw.loc[features_complete_mask, [V_COL, P_COL]].copy()
c4["alignment_key"] = c4.index.astype(int)
candidates["features_complete_original_index"] = c4

# Candidate 5: complete cases in features only, reset index
c5 = target_raw.loc[features_complete_mask, [V_COL, P_COL]].copy().reset_index(drop=True)
c5["alignment_key"] = np.arange(len(c5))
candidates["features_complete_reset_index"] = c5

# Candidate 6: target complete only, reset index
target_complete_mask = target_raw[[V_COL, P_COL]].replace([np.inf, -np.inf], np.nan).notna().all(axis=1)

c6 = target_raw.loc[target_complete_mask, [V_COL, P_COL]].copy().reset_index(drop=True)
c6["alignment_key"] = np.arange(len(c6))
candidates["target_complete_reset_index"] = c6


print("\n=== CANDIDATE ALIGNMENT SIZES ===")
for name, cand in candidates.items():
    print(name, cand.shape, "key min/max:", cand["alignment_key"].min(), cand["alignment_key"].max())


# ------------------------------------------------------------
# Score alignments
# ------------------------------------------------------------

score_rows = []

for cand_name, cand in candidates.items():
    for outcome, pred in predictions.items():

        merged = pred.merge(
            cand,
            left_on="idx_original",
            right_on="alignment_key",
            how="left"
        )

        if merged[[V_COL, P_COL]].isna().any().any():
            match_rate = np.nan
            positives_calc = np.nan
            cm_calc = {"TP": np.nan, "FP": np.nan, "TN": np.nan, "FN": np.nan}
        else:
            y_calc = make_y_from_counts(merged, outcome, threshold=1)
            match_rate = float((y_calc.to_numpy() == merged["y_true_official"].to_numpy()).mean())
            positives_calc = int(y_calc.sum())
            cm_calc = cm_dict(y_calc, merged["y_pred"])

        score_rows.append({
            "candidate": cand_name,
            "outcome": outcome,
            "match_rate_ytrue_ge1": match_rate,
            "official_positives": int(pred["y_true_official"].sum()),
            "calculated_positives_ge1": positives_calc,
            **cm_calc
        })

score_df = pd.DataFrame(score_rows)

print("\n=== ALIGNMENT SCORE ===")
print(score_df.to_string(index=False))

score_df.to_csv(OUTPUT_DIR / "alignment_score_candidates.csv", index=False)

# Mean match by candidate
mean_score = (
    score_df
    .groupby("candidate", as_index=False)["match_rate_ytrue_ge1"]
    .mean()
    .sort_values("match_rate_ytrue_ge1", ascending=False)
)

print("\n=== MEAN ALIGNMENT SCORE ===")
print(mean_score.to_string(index=False))

best_candidate = mean_score.iloc[0]["candidate"]
best_match = mean_score.iloc[0]["match_rate_ytrue_ge1"]

print("\nBEST CANDIDATE:", best_candidate)
print("BEST MEAN MATCH:", best_match)

if best_match < 0.999:
    raise ValueError(
        "No candidate alignment reproduced official y_true >=1. "
        "Do NOT compute threshold sensitivity yet. "
        "Need to export V.SUM.TOTAL/P.SUM.TOTAL from inside the final notebooks."
    )

aligned_counts = candidates[best_candidate].copy()


# ------------------------------------------------------------
# Compute threshold sensitivity using aligned counts
# ------------------------------------------------------------

thresholds = [1, 2, 3]
rows = []
rowlevel_outputs = {}

for outcome, pred in predictions.items():

    merged = pred.merge(
        aligned_counts,
        left_on="idx_original",
        right_on="alignment_key",
        how="left"
    )

    if merged[[V_COL, P_COL]].isna().any().any():
        raise ValueError(f"Missing aligned counts for {outcome}")

    # validation at >=1
    y_check = make_y_from_counts(merged, outcome, threshold=1)
    match_rate = (y_check.to_numpy() == merged["y_true_official"].to_numpy()).mean()

    print("\n======================================")
    print("Outcome:", outcome)
    print("Validation match official y_true >=1:", match_rate)
    print("Official CM:", cm_dict(merged["y_true_official"], merged["y_pred"]))
    print("Aligned >=1 CM:", cm_dict(y_check, merged["y_pred"]))

    rowlevel_outputs[outcome] = merged.copy()

    for t in thresholds:
        y_alt = make_y_from_counts(merged, outcome, threshold=t)

        if outcome == "victimization":
            alt_definition = f"{V_COL} >= {t}"
        elif outcome == "perpetration":
            alt_definition = f"{P_COL} >= {t}"
        else:
            alt_definition = f"{V_COL} >= {t} AND {P_COL} >= {t}"

        metrics = binary_metrics(y_alt, merged["y_pred"])

        rows.append({
            "outcome": outcome,
            "severity_threshold": f">={t}",
            "alternative_definition": alt_definition,
            "alignment_used": best_candidate,
            **metrics
        })


sensitivity_df = pd.DataFrame(rows)

main_csv = OUTPUT_DIR / "threshold_sensitivity_ge1_ge2_ge3_final_PCA_trainonly_ALIGNED.csv"
pretty_csv = OUTPUT_DIR / "threshold_sensitivity_ge1_ge2_ge3_final_PCA_trainonly_ALIGNED_pretty.csv"

sensitivity_df.to_csv(main_csv, index=False)

for outcome, merged in rowlevel_outputs.items():
    merged.to_csv(OUTPUT_DIR / f"{outcome}_threshold_sensitivity_rowlevel_ALIGNED.csv", index=False)

pretty = sensitivity_df.copy()

metric_cols = [
    "recall_sensitivity",
    "specificity",
    "precision_ppv",
    "npv",
    "balanced_accuracy",
    "f1_positive",
    "accuracy",
    "fpr",
    "fnr"
]

for c in metric_cols:
    pretty[c] = pretty[c].apply(lambda x: f"{x*100:.1f}%" if pd.notna(x) else "")

pretty.to_csv(pretty_csv, index=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 2400)

display_cols = [
    "outcome",
    "severity_threshold",
    "alternative_definition",
    "alignment_used",
    "n",
    "positives",
    "negatives",
    "TP",
    "FP",
    "TN",
    "FN",
    "recall_sensitivity",
    "specificity",
    "precision_ppv",
    "npv",
    "balanced_accuracy",
    "f1_positive",
    "accuracy",
    "fpr",
    "fnr"
]

print("\n=== THRESHOLD SENSITIVITY >=1 / >=2 / >=3 — ALIGNED ===")
print(
    sensitivity_df[display_cols].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print("\n=== PRETTY — ALIGNED ===")
print(pretty[display_cols].to_string(index=False))

print("\nSaved to:")
print(OUTPUT_DIR.resolve())

features_raw: (4024, 29)
target_raw: (4024, 11)

 victimization
n: 942
idx min/max: 2 3766
official positives: 465
official CM: {'TP': 405, 'FP': 329, 'TN': 148, 'FN': 60}

 perpetration
n: 942
idx min/max: 1 3759
official positives: 221
official CM: {'TP': 202, 'FP': 504, 'TN': 217, 'FN': 19}

 overlap
n: 942
idx min/max: 1 3759
official positives: 178
official CM: {'TP': 144, 'FP': 355, 'TN': 409, 'FN': 34}

=== CANDIDATE ALIGNMENT SIZES ===
raw_direct_index (4024, 3) key min/max: 0 4023
complete_cases_original_index (4024, 3) key min/max: 0 4023
complete_cases_reset_index (4024, 3) key min/max: 0 4023
features_complete_original_index (4024, 3) key min/max: 0 4023
features_complete_reset_index (4024, 3) key min/max: 0 4023
target_complete_reset_index (4024, 3) key min/max: 0 4023

=== ALIGNMENT SCORE ===
                       candidate       outcome  match_rate_ytrue_ge1  official_positives  calculated_positives_ge1  TP  FP  TN  FN
                raw_direct_index victimization     

ValueError: No candidate alignment reproduced official y_true >=1. Do NOT compute threshold sensitivity yet. Need to export V.SUM.TOTAL/P.SUM.TOTAL from inside the final notebooks.